In [ ]:
# Install python-chess, torch, numpy

In [9]:
import chess
import chess.pgn
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

In [2]:
def board_to_tensor(board):
    # Initialize tensor with zeros (12 planes: 6 piece types × 2 colors)
    tensor = torch.zeros(12, 8, 8)
    
    # Map piece symbols to channel indices
    piece_to_index = {
        'P': 0, 'N': 1, 'B': 2, 'R': 3, 'Q': 4, 'K': 5,  # White pieces
        'p': 6, 'n': 7, 'b': 8, 'r': 9, 'q': 10, 'k': 11  # Black pieces
    }
    
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece:
            piece_type = piece.symbol()
            channel = piece_to_index[piece_type]

            row = chess.square_rank(square)
            col = chess.square_file(square)

            tensor[channel, row, col] = 1
    
    return tensor

In [3]:
class ChessBot(nn.Module):
    def __init__(self):
        super(ChessBot, self).__init__()
        
        self.conv1 = nn.Conv2d(12, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 8 * 8, 1024)
        self.fc2 = nn.Linear(1024, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [4]:
def get_legal_moves(board):
    return list(board.legal_moves)

def make_move(board, move):
    board = board.copy()
    board.push(move)
    return board

def is_game_over(board):
    return board.is_game_over()

def evaluate_position(board, model):
    # Check for game-over states
    if board.is_checkmate():
        # If it's white's turn and checkmate, white lost (black wins)
        if board.turn == chess.WHITE:
            return -1000
        else:
            return 1000
    if board.is_stalemate() or board.is_insufficient_material():
        return 0
    
    # Convert board to tensor and add batch dimension
    tensor = board_to_tensor(board)
    tensor = tensor.unsqueeze(0)
    
    # Get neural network evaluation
    with torch.no_grad():
        score = model(tensor)
    
    return score.item()

In [5]:
def minimax(board, depth, alpha, beta, maximizing_player, model):
    if depth == 0 or board.is_game_over():
        return evaluate_position(board, model), None
    
    if maximizing_player:
        max_eval = float('-inf')
        best_move = None
        for move in get_legal_moves(board):
            new_board = make_move(board, move)
            eval_score, _ = minimax(new_board, depth - 1, alpha, beta, False, model)
            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move
            alpha = max(alpha, eval_score)
            if beta <= alpha:
                break
        return max_eval, best_move
    else:
        min_eval = float('inf')
        best_move = None
        for move in get_legal_moves(board):
            new_board = make_move(board, move)
            eval_score, _ = minimax(new_board, depth - 1, alpha, beta, True, model)
            if eval_score < min_eval:
                min_eval = eval_score
                best_move = move
            beta = min(beta, eval_score)
            if beta <= alpha:
                break
        return min_eval, best_move

In [ ]:
file_path = "E:\\Projects\\ChessBot\\Data\\lichess_elite_2021-10.pgn"

def parse_pgn_file(file_path):
    """Parse PGN file and yield games one by one"""
    with open(file_path, 'r') as f:
        while True:
            game = chess.pgn.read_game(f)
            if game is None:
                break
            yield game

def extract_training_data(game):
    """Extract positions and game result from a single game"""
    positions = []
    board = game.board()
    
    # Get the game result from headers
    result = game.headers['Result']
    # Fill in: Map result to numeric value (+1, -1, or 0)
    # Hint: "1-0" means white won, "0-1" means black won, "1/2-1/2" is draw
    if result == "1-0":
        game_result = 1  # White won
    elif result == "0-1":
        game_result = -1  # Black won
    else:
        game_result = 0 # Draw
    
    # Walk through all moves in the game
    for move in game.mainline_moves():
        # Fill in: Convert current board to tensor
        tensor = board_to_tensor(board) # Use your board_to_tensor function
        positions.append((tensor, game_result))
        
        # Fill in: Make the move on the board
        board.push(move)
    
    return positions

def create_dataset(pgn_file, max_games=800):
    """Create training dataset from PGN file"""
    dataset = []
    game_count = 0
    
    for game in parse_pgn_file(pgn_file):
        if game_count >= max_games:
            break
        positions = extract_training_data(game)
        dataset.extend(positions)
        game_count += 1
        if game_count % 100 == 0:
            print(f"Processed {game_count} games...")
    
    return dataset

In [16]:
# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA device count: {torch.cuda.device_count()}")
else:
    print("CUDA not available - will use CPU")

def train_model(dataset, epochs=15, batch_size=64, learning_rate=0.001):
    model = ChessBot()
    
    # Use GPU if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    print(f"Training on: {device}")
    
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    
    # Initialize best loss to infinity
    best_loss = float('inf')
    
    for epoch in range(epochs):
        random.shuffle(dataset)
        total_loss = 0
        
        for i in range(0, len(dataset), batch_size):
            batch = dataset[i:i+batch_size]
            boards = torch.stack([item[0] for item in batch]).to(device)
            targets = torch.tensor([item[1] for item in batch], dtype=torch.float32).unsqueeze(1).to(device)
            
            optimizer.zero_grad()
            predictions = model(boards)
            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataset)
        print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
        
        # Check if current loss is better than best loss
        if avg_loss < best_loss:
            best_loss = avg_loss
            # Save the model (move back to CPU before saving)
            torch.save(model.cpu().state_dict(), 'best_chess_model.pth')
            model = model.to(device)  # Move back to GPU
            print(f"  -> New best model saved! Loss: {best_loss:.4f}")
    
    return model

CUDA available: True
CUDA device: NVIDIA GeForce RTX 3050 A Laptop GPU
CUDA device count: 1


In [17]:
# Create smaller dataset for testing
dataset = create_dataset(file_path, max_games=800)

# Train model with larger batch size for better GPU utilization
model = train_model(dataset, epochs=15, batch_size=64, learning_rate=0.001)

Processed 100 games...
Processed 200 games...
Processed 300 games...
Processed 400 games...
Processed 500 games...
Processed 600 games...
Processed 700 games...
Processed 800 games...
Training on: cuda
Epoch 1, Loss: 0.0114
  -> New best model saved! Loss: 0.0114
Epoch 2, Loss: 0.0051
  -> New best model saved! Loss: 0.0051
Epoch 3, Loss: 0.0033
  -> New best model saved! Loss: 0.0033
Epoch 4, Loss: 0.0028
  -> New best model saved! Loss: 0.0028
Epoch 5, Loss: 0.0024
  -> New best model saved! Loss: 0.0024
Epoch 6, Loss: 0.0023
  -> New best model saved! Loss: 0.0023
Epoch 7, Loss: 0.0021
  -> New best model saved! Loss: 0.0021
Epoch 8, Loss: 0.0020
  -> New best model saved! Loss: 0.0020
Epoch 9, Loss: 0.0019
  -> New best model saved! Loss: 0.0019
Epoch 10, Loss: 0.0019
  -> New best model saved! Loss: 0.0019
Epoch 11, Loss: 0.0018
  -> New best model saved! Loss: 0.0018
Epoch 12, Loss: 0.0018
  -> New best model saved! Loss: 0.0018
Epoch 13, Loss: 0.0017
  -> New best model saved! L